# 🎯 03 - Indexing and Selection

Time to slice and dice.

In this notebook, you'll:
- Use `.loc[]` for label-based selection, and `.iloc[]` for position-based selection
- Create boolean masks to filter rows
- Select specific rows **and** columns together in one step
- Change values safely, without the "chained assignment" trap

## ✅ Learning Goals
- Tell `.loc[]` and `.iloc[]` apart, and know when to use each
- Select specific rows and columns with lists of labels or positions
- Give a DataFrame a meaningful index with `.set_index()`
- Grab a single value with `.at[]` and `.iat[]`
- Filter rows with a boolean mask, combine conditions with `&` and `|`, and flip one with `~`
- Match several values with `.isin()`, a range with `.between()`, and missing values with `.isna()`/`.notna()`
- Know when a selection gives you a `Series` and when it gives you a `DataFrame`
- Select rows and columns together with `df.loc[condition, columns]`, and change values without chained assignment

> **The core habit:** use `.loc[]` when the labels matter, `.iloc[]` when positions matter, and boolean masks when conditions matter.

In [1]:
import pandas as pd

df = pd.read_csv('titanic.csv')
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


Both selectors use the same shape: **rows first, then columns**, separated by a comma.

```python
df.loc[row_labels, column_labels]          # by label (names)
df.iloc[row_positions, column_positions]   # by position (0, 1, 2, ...)
```

Leave out the columns to get every column. A bare `:` also means "all", so `df.loc[:, ['Age']]` is every row of the `Age` column.

## 🔎 Select Rows by Label with `.loc[]`

In [2]:
# .loc[row_label, column_label]
# Titanic uses the default RangeIndex, so labels here happen to be integers
print(df.loc[0])

PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
Name: 0, dtype: object


In [3]:
# Row range (inclusive on both ends!) with specific columns
df.loc[0:4, ['Age', 'Fare', 'Sex']]

,Age,Fare,Sex
0,22.0,7.2500,male
1,38.0,71.2833,female
2,26.0,7.9250,female
3,35.0,53.1000,female
4,35.0,8.0500,male


To pick specific rows that aren't next to each other, pass a **list** of labels:

In [4]:
df.loc[[0, 2, 4], ['Name', 'Age']]

,Name,Age
0,"Braund, Mr. Owen Harris",22.0
2,"Heikkinen, Miss. Laina",26.0
4,"Allen, Mr. William Henry",35.0


## 🔢 Select Rows by Position with `.iloc[]`

In [5]:
df.iloc[0]  # First row, by position

PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
Name: 0, dtype: object

In [6]:
# Row slice, column slice — both by position, stop-exclusive like a normal Python slice
df.iloc[1:4, 0:5]

,PassengerId,Survived,Pclass,Name,Sex
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female
2,3,1,3,"Heikkinen, Miss. Laina",female
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female


`.iloc[]` takes lists too, of **positions**. Columns 3 and 5 are `Name` and `Age`:

In [7]:
df.iloc[[0, 2, 4], [3, 5]]

,Name,Age
0,"Braund, Mr. Owen Harris",22.0
2,"Heikkinen, Miss. Laina",26.0
4,"Allen, Mr. William Henry",35.0


### 🧠 Quick Check
`.loc[0:4]` includes row `4`. `.iloc[0:4]` does **not** include position `4`. Why do you think `.loc[]` breaks the usual "stop-exclusive" slicing rule you learned for lists and strings? (Hint: think about what happens when labels aren't numbers at all, like `.loc['Alice':'Charlie']`.)

## 🏷️ A Meaningful Index with `.set_index()`
Row labels `0, 1, 2, ...` don't mean much. `.set_index()` turns a column into the row labels, so `.loc[]` can look rows up by something meaningful. (`.iloc[]` is unaffected: it always uses positions.)

In [8]:
by_id = df.set_index('PassengerId')
by_id.loc[2, ['Name', 'Age', 'Fare']]   # passenger #2, not the third row

Name    Cumings, Mrs. John Bradley (Florence Briggs Th...
Age                                                  38.0
Fare                                              71.2833
Name: 2, dtype: object

In [9]:
by_name = df.set_index('Name')
by_name.loc['Braund, Mr. Owen Harris', 'Age']

np.float64(22.0)

### When `.loc[]` and `.iloc[]` Disagree
With the default `0, 1, 2, ...` index, a row's label and its position are the same number, so `.loc[3]` and `.iloc[3]` look interchangeable. Once the index means something, they don't:

In [10]:
print('by_id.loc[3]  →', by_id.loc[3, 'Name'])    # the row LABELED 3: passenger #3
print('by_id.iloc[3] →', by_id.iloc[3]['Name'])   # the row at POSITION 3: the 4th row

by_id.loc[3]  → Heikkinen, Miss. Laina
by_id.iloc[3] → Futrelle, Mrs. Jacques Heath (Lily May Peel)


| | `.loc[]` | `.iloc[]` |
|---|---|---|
| Selects using | **Labels** (index values and column names) | **Positions** (`0, 1, 2, ...`) |
| Example | `df.loc[2, 'Fare']` | `df.iloc[2, 9]` |
| End of a slice | **Included** (`.loc[1:3]` gives 3 rows) | **Excluded** (`.iloc[1:3]` gives 2 rows) |
| Columns chosen by | Name | Position |
| Best for | Readable, named selections | "The first 5 rows", "the last column" |

`.set_index()` returns a **new** DataFrame, so `df` itself still has its original `0, 1, 2, ...` index.

## 🎯 Grabbing One Value: `.at[]` and `.iat[]`
`.loc[]` and `.iloc[]` can return a single value, but `.at[]` (by label) and `.iat[]` (by position) are made for exactly that, and are faster:

In [11]:
print(df.loc[1, 'Fare'])   # label-based, general purpose
print(df.iloc[1, 9])       # position-based: row 1, column 9 (Fare)
print(df.at[1, 'Fare'])    # label-based, ONE value
print(df.iat[1, 9])        # position-based, ONE value

71.2833
71.2833
71.2833
71.2833


## 🎯 Boolean Masking (Filtering)

A condition on a column doesn't return rows. It returns a **mask**: a `Series` of `True`/`False`, one per row:

In [12]:
under_10 = df['Age'] < 10
print(under_10.head(8))
print('How many are True:', under_10.sum())   # True counts as 1, False as 0

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7     True
Name: Age, dtype: bool
How many are True: 62


Putting the mask inside `df[...]` keeps only the rows where it's `True`. `df.loc[mask]` does exactly the same thing:

In [13]:
# Filter passengers under age 10
kids = df[df['Age'] < 10]
kids[['Age', 'Sex', 'Pclass']].head()

,Age,Sex,Pclass
7,2.0,male,3
10,4.0,female,3
16,2.0,male,3
24,8.0,female,3
43,3.0,female,2


## 🧠 Compound Conditions
Use `&` for "and" and `|` for "or" — **not** the `and`/`or` keywords from Control Flow. Each condition also needs its own parentheses.

In [14]:
# Adults in First Class who paid over $100
rich_folks = df[(df['Age'] > 18) & (df['Pclass'] == 1) & (df['Fare'] > 100)]
rich_folks[['Age', 'Pclass', 'Fare']].head()

,Age,Pclass,Fare
27,19.0,1,263.0000
88,23.0,1,263.0000
118,24.0,1,247.5208
195,58.0,1,146.5208
215,31.0,1,113.2750


### Why Not `and` / `or`?
`df[(df['Age'] < 10) and (df['Pclass'] == 1)]` fails with:

```text
ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
```

Python's `and` wants **one** True or False, but a mask holds **hundreds** of them, and Python can't tell which one you mean. `&` and `|` combine two masks **row by row**, which is what you want. Forgetting the parentheses causes a similar error, because `&` is evaluated before `<` and `==`.

## 🧺 Matching Several Values: `.isin()` and `.between()`
Checking one column against a **list** of values with `|` gets long fast. `.isin()` does it in one step, and `.between()` checks a range. Both ends are **included**:

In [15]:
# Instead of (df['Embarked'] == 'C') | (df['Embarked'] == 'Q')
cherbourg_or_queenstown = df[df['Embarked'].isin(['C', 'Q'])]
print('Embarked at C or Q:', len(cherbourg_or_queenstown))

teens = df[df['Age'].between(13, 19)]      # 13 <= Age <= 19
print('Teenagers:', len(teens))

Embarked at C or Q: 245
Teenagers: 95


### Flipping a Condition with `~`
`~` means **not**. It turns every `True` into `False` and every `False` into `True`. For a simple comparison, `!=` is just as good, but `~` is the only way to say "not" with `.isin()`, `.between()`, or `.isna()`:

In [16]:
not_southampton = df[~df['Embarked'].isin(['S'])]
print('Did NOT board at Southampton:', len(not_southampton))

Did NOT board at Southampton: 247


### Missing Values as a Mask
`.isna()` is `True` where a value is missing, and `.notna()` is the opposite:

In [17]:
print('Age missing:', df['Age'].isna().sum())
print('Age known:  ', df['Age'].notna().sum())
df[df['Age'].isna()][['Name', 'Age']].head(3)

Age missing: 177
Age known:   714


,Name,Age
5,"Moran, Mr. James",NaN
17,"Williams, Mr. Charles Eugene",NaN
19,"Masselmani, Mrs. Fatima",NaN


💡 Text columns have their own masks too, like `df['Name'].str.contains('Mrs')`. Notebook 06 covers those.

## 🧹 Select Columns Only

In [18]:
df[['Sex', 'Age', 'Embarked']].head()

,Sex,Age,Embarked
0,male,22.0,S
1,female,38.0,C
2,female,26.0,S
3,female,35.0,S
4,male,35.0,S


### Series or DataFrame?
**Single** brackets with one name give you a `Series`. **Double** brackets (a list of names, even a list of one) give you a `DataFrame`:

In [19]:
print(type(df['Age']))      # Series
print(type(df[['Age']]))    # DataFrame (a table with one column)

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


You may also see `df.Age` (attribute notation). It works for simple names, but **prefer brackets**. They work for every column name, including ones with spaces (`df['order total']`) and ones that clash with DataFrame methods (a column named `count` or `size`).

## 🎯 Rows and Columns Together
The single most useful selection pattern combines a **mask for the rows** with a **list of columns**, in one `.loc[]`:

In [20]:
df.loc[df['Pclass'] == 1, ['Name', 'Age', 'Fare']].head()

,Name,Age,Fare
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000
6,"McCarthy, Mr. Timothy J",54.0,51.8625
11,"Bonnell, Miss. Elizabeth",58.0,26.5500
23,"Sloper, Mr. William Thompson",28.0,35.5000


Read it as: *"From `df`, keep the rows where `Pclass` is 1, then return `Name`, `Age`, and `Fare`."* The row condition can be as complex as you need:

In [21]:
df.loc[(df['Sex'] == 'female') & (df['Fare'] < 10), ['Name', 'Age', 'Fare']].head()

,Name,Age,Fare
2,"Heikkinen, Miss. Laina",26.0,7.9250
14,"Vestrom, Miss. Hulda Amanda Adolfina",14.0,7.8542
19,"Masselmani, Mrs. Fatima",NaN,7.2250
22,"McGowan, Miss. Anna ""Annie""",15.0,8.0292
28,"O'Dwyer, Miss. Ellen ""Nellie""",NaN,7.8792


### The Top or Bottom N
To find the rows with the largest or smallest values in a column, `.nlargest()` and `.nsmallest()` are shortcuts for sorting and then taking `.head()`:

In [22]:
df.nlargest(5, 'Fare')[['Name', 'Pclass', 'Fare']]

,Name,Pclass,Fare
258,"Ward, Miss. Anna",1,512.3292
679,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292
737,"Lesurer, Mr. Gustave J",1,512.3292
27,"Fortune, Mr. Charles Alexander",1,263.0000
88,"Fortune, Miss. Mabel Helen",1,263.0000


## ✏️ Changing Values: Avoid Chained Assignment
Selecting and then assigning in **two** bracket steps looks reasonable, but it doesn't work. The first step makes a **copy**, so the assignment changes the copy and then throws it away. Pandas warns you, but the original is untouched:

In [23]:
# ❌ Two steps: filter, THEN assign to the filtered copy
df[df['Age'] < 10]['child'] = True

print('Did it add a "child" column?', 'child' in df.columns)

Did it add a "child" column? False


C:\Users\zucum\AppData\Local\Temp\ipykernel_41392\3559352248.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[df['Age'] < 10]['child'] = True


Do it in **one** `.loc[]` step instead. `.loc[]` names the original DataFrame, the rows, and the column all at once. Give every row a default first, so the rows that don't match aren't left as `NaN` (notebook 05 uses this pattern to build new columns):

In [24]:
# ✅ One step, with .loc[rows, column]
df['child'] = False
df.loc[df['Age'] < 10, 'child'] = True

df['child'].value_counts()

child
False    829
True      62
Name: count, dtype: int64

💡 If you make a filtered DataFrame that you plan to **change**, add `.copy()`, as in `kids = df[df['Age'] < 10].copy()`. That makes it clear you mean a separate table, and pandas won't warn you.

## 🔢 Resetting the Index After Filtering
A filtered or sorted DataFrame keeps its **original** row labels, so they're no longer `0, 1, 2, ...`:

In [25]:
kids = df[df['Age'] < 10]
print('Labels after filtering:', kids.index[:5].tolist())

kids = kids.reset_index(drop=True)
print('Labels after reset:    ', kids.index[:5].tolist())

Labels after filtering: [7, 10, 16, 24, 43]
Labels after reset:     [0, 1, 2, 3, 4]


`drop=True` throws the old labels away instead of keeping them as a new column. Don't reset out of habit: if the index **means** something (like `PassengerId`), resetting loses it.

## 🧭 A Selection Workflow
Before writing a selection, ask yourself:
1. Do I need **rows**, **columns**, or both?
2. Am I selecting by **label** or by **position**?
3. Do I need a **condition**?
4. Should the result be a `Series` or a `DataFrame`?
5. Am I **changing** the original data? (If so, use one `.loc[]` step.)

Most real selections end up looking like this: a condition, a column list, and maybe a sort:

In [26]:
(df.loc[(df['Pclass'] == 3) & (df['Survived'] == 1), ['Name', 'Age', 'Fare']]
   .sort_values('Fare', ascending=False)
   .head())

,Name,Age,Fare
74,"Bing, Mr. Lee",32.0,56.4958
838,"Chip, Mr. Chang",32.0,56.4958
509,"Lang, Mr. Fang",26.0,56.4958
643,"Foo, Mr. Choong",NaN,56.4958
692,"Lam, Mr. Ali",NaN,56.4958


## 🏋️ Practice

### Practice 1 — Basic Selection (easy)
Select the first row using `.iloc[]`, then again using `.loc[]`. Confirm they match.

In [27]:
# Your code here
first_row_iloc = df.iloc[0]
first_row_loc = df.loc[df.index[0]]
print(first_row_iloc)
print(first_row_loc)
print(first_row_iloc.equals(first_row_loc))

PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
child                            False
Name: 0, dtype: object
PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
ch

### Practice 2 — Filter and Select (medium)
Filter the dataset for female passengers over age 40, showing only the `Name`, `Age`, and `Pclass` columns.

In [28]:
# Your code here
filtered_df = df.loc[(df['Sex'] == 'female') & (df['Age'] > 40), ['Name', 'Age', 'Pclass']]
print(filtered_df)

                                                  Name   Age  Pclass
11                            Bonnell, Miss. Elizabeth  58.0       1
15                    Hewlett, Mrs. (Mary D Kingcome)   55.0       2
52            Harper, Mrs. Henry Sleeper (Myna Haxtun)  49.0       1
132     Robins, Mrs. Alexander A (Grace Charity Laury)  47.0       3
167    Skoog, Mrs. William (Anna Bernhardina Karlsson)  45.0       3
177                         Isham, Miss. Ann Elizabeth  50.0       1
194          Brown, Mrs. James Joseph (Margaret Tobin)  44.0       1
195                               Lurette, Miss. Elise  58.0       1
254           Rosblom, Mrs. Viktor (Helena Wilhelmina)  41.0       3
259                        Parrish, Mrs. (Lutie Davis)  50.0       2
268      Graham, Mrs. William Thompson (Edith Junkins)  58.0       1
272          Mellinger, Mrs. (Elizabeth Anne Maidment)  41.0       2
275                  Andrews, Miss. Kornelia Theodosia  63.0       1
276                  Lindblom, Mis

### Practice 3 — Missing Data Slice (harder)
Find passengers with a missing `Age` (hint: `.isnull()`) and print only their `Name`, `Age`, and `Pclass` columns.

In [29]:
# Your code here
missing_age_df = df.loc[df['Age'].isnull(), ['Name', 'Age', 'Pclass']]
print(missing_age_df)

                                         Name  Age  Pclass
5                            Moran, Mr. James  NaN       3
17               Williams, Mr. Charles Eugene  NaN       2
19                    Masselmani, Mrs. Fatima  NaN       3
26                    Emir, Mr. Farred Chehab  NaN       3
28              O'Dwyer, Miss. Ellen "Nellie"  NaN       3
..                                        ...  ...     ...
859                          Razi, Mr. Raihed  NaN       3
863         Sage, Miss. Dorothy Edith "Dolly"  NaN       3
868               van Melkebeke, Mr. Philemon  NaN       3
878                        Laleff, Mr. Kristo  NaN       3
888  Johnston, Miss. Catherine Helen "Carrie"  NaN       3

[177 rows x 3 columns]


### Practice 4 — `.isin()` and `.between()` (medium)
Find passengers who embarked at Cherbourg (`'C'`) or Queenstown (`'Q'`) **and** paid a fare between \$50 and \$100. Show only `Name`, `Embarked`, and `Fare`. How many are there?

In [30]:
# Your code here
embarked_cq_and_fare_between = df.loc[(df['Embarked'].isin(['C', 'Q'])) & (df['Fare'].between(50, 100)), :]
print(embarked_cq_and_fare_between)
print(embarked_cq_and_fare_between.shape[0])

     PassengerId  Survived  Pclass  \
1              2         1       1   
34            35         0       1   
52            53         1       1   
54            55         0       1   
97            98         1       1   
139          140         0       1   
155          156         0       1   
218          219         1       1   
245          246         0       1   
256          257         1       1   
291          292         1       1   
309          310         1       1   
310          311         1       1   
329          330         1       1   
366          367         1       1   
369          370         1       1   
370          371         1       1   
375          376         1       1   
412          413         1       1   
453          454         1       1   
484          485         1       1   
496          497         1       1   
513          514         1       1   
523          524         1       1   
587          588         1       1   
591         

### Practice 5 — Not and Not Missing (medium)
Using `~` and `.notna()`, find passengers who did **not** travel in 3rd class **and** whose `Age` is known. Show only `Name`, `Pclass`, and `Age` with a single `.loc[]`, then reset the index so the rows are numbered from 0.

In [31]:
# Your code here
not_third_class_and_known_age = df.loc[(~(df['Pclass'] == 3)) & (df['Age'].notna()), ['Name', 'Pclass', 'Age']].reset_index(drop=True)
print(not_third_class_and_known_age)
print(not_third_class_and_known_age.shape[0])

                                                  Name  Pclass   Age
0    Cumings, Mrs. John Bradley (Florence Briggs Th...       1  38.0
1         Futrelle, Mrs. Jacques Heath (Lily May Peel)       1  35.0
2                              McCarthy, Mr. Timothy J       1  54.0
3                  Nasser, Mrs. Nicholas (Adele Achem)       2  14.0
4                             Bonnell, Miss. Elizabeth       1  58.0
..                                                 ...     ...   ...
354       Shelley, Mrs. William (Imanita Parrish Hall)       2  25.0
355                      Banfield, Mr. Frederick James       2  28.0
356                              Montvila, Rev. Juozas       2  27.0
357                       Graham, Miss. Margaret Edith       1  19.0
358                              Behr, Mr. Karl Howell       1  26.0

[359 rows x 3 columns]
359


## ✏️ Your Turn
1. Slice the first 5 rows and select only `Survived`, `Sex`, and `Age`.
2. Create a DataFrame of passengers in 2nd class under age 15.
3. Use `.loc[]` to select rows 10–20 and only the columns `Fare`, `Pclass`, and `Age`. Then use `.iloc[]` to get the **same** rows and columns by position.
4. Make a copy indexed by `Name` with `.set_index()`, then use `.at[]` to look up one passenger's `Fare`.
5. Show the `Name` and `Age` of the 5 **youngest** passengers with `.nsmallest()`.
6. Add an `age_unknown` column that's `True` for passengers with a missing `Age` and `False` otherwise, using `.loc[]` (no chained assignment!).
7. In a comment, explain why `df.loc[1:3]` returns 3 rows but `df.iloc[1:3]` returns 2.

In [32]:
# Your turn here
# 1. Slice the first 5 rows and select only `Survived`, `Sex`, and `Age`.
print(df.loc[:4, ['Survived', 'Sex', 'Age']])

# 2. Create a DataFrame of passengers in 2nd class under age 15.
second_class_under_15 = df.loc[(df['Pclass'] == 2) & (df['Age'] < 15)]
print(second_class_under_15)

# 3. Use `.loc[]` to select rows 10–20 and only the columns `Fare`, `Pclass`, and `Age`. Then use `.iloc[]` to get the **same** rows and columns by position.
print(df.loc[10:20, ['Fare', 'Pclass', 'Age']])
print(df.iloc[10:21, [df.columns.get_loc('Fare'), df.columns.get_loc('Pclass'), df.columns.get_loc('Age')]])

# 4. Make a copy indexed by `Name` with `.set_index()`, then use `.at[]` to look up one passenger's `Fare`.
df_by_name = df.set_index('Name').copy()
print(df_by_name.at['Braund, Mr. Owen Harris', 'Fare'])

# 5. Show the `Name` and `Age` of the 5 **youngest** passengers with `.nsmallest()`.
print(df.nsmallest(5, 'Age')[['Name', 'Age']])

# 6. Add an `age_unknown` column that's `True` for passengers with a missing `Age` and `False` otherwise, using `.loc[]` (no chained assignment!).
df.loc[:, 'age_unknown'] = df['Age'].isna()

# 7. In a comment, explain why `df.loc[1:3]` returns 3 rows but `df.iloc[1:3]` returns 2.
# `df.loc[1:3]` includes both the start and end labels (1, 2, 3), while `df.iloc[1:3]` uses integer positions and is exclusive of the stop index.

   Survived     Sex   Age
0         0    male  22.0
1         1  female  38.0
2         1  female  26.0
3         1  female  35.0
4         0    male  35.0
     PassengerId  Survived  Pclass                                      Name  \
9             10         1       2       Nasser, Mrs. Nicholas (Adele Achem)   
43            44         1       2  Laroche, Miss. Simonne Marie Anne Andree   
58            59         1       2              West, Miss. Constance Mirium   
78            79         1       2             Caldwell, Master. Alden Gates   
183          184         1       2                 Becker, Master. Richard F   
193          194         1       2                Navratil, Master. Michel M   
237          238         1       2          Collyer, Miss. Marjorie "Lottie"   
340          341         1       2            Navratil, Master. Edmond Roger   
407          408         1       2            Richards, Master. William Rowe   
446          447         1       2         M

## 🔥 Challenge (Optional) — One Expression
Find the **10 youngest female survivors in 1st or 2nd class**, showing only `Name`, `Age`, and `Pclass`, sorted from youngest to oldest, with rows numbered `0`–`9`.

Do it as **one** chained expression: a `.loc[]` with a compound condition (use `.isin()` for the classes) and a column list, then `.sort_values()`, `.head()`, and `.reset_index(drop=True)`. Passengers with a missing `Age` shouldn't appear. Does `.sort_values()` put them last, or do you need to filter them out?

In [33]:
# Your code here


---
## 🧼 Why This Matters
Being able to surgically select parts of your dataset is the core of real analysis. Whether you're cleaning junk or feeding a model, proper indexing is power.

The most flexible everyday pattern is `df.loc[row_condition, ['column_a', 'column_b']]`. It reads naturally, selects exactly what you need, and works for both looking at data and changing it.

Next: cleaning your data with `.dropna()`, `.fillna()`, and more.